> **Public-copy note:** Saved cell outputs were cleared because some belonged to an earlier evaluation run. This notebook was prepared for Google Colab and expects the dataset and model checkpoint in the configured paths. It has not been executed as part of this repository. Review the remote model code before running cells that use `trust_remote_code=True`.


# مشروع RafiQ — تدريب وتقييم النموذج

هذا النوتبوك مرتب لشرح مراحل المشروع بشكل واضح من **تحميل الداتا سيت** وحتى **التقييم النهائي والمسار الصوتي**.

⭐ **الخلايا المعلّمة بعبارة «للحكّام» هي أهم الخلايا التي تُظهر الفرق بين أداء النموذج قبل التدريب وبعده، إضافة إلى نتيجة الديمو الصوتي.**


# 🩺 RafiQ — تطبيع اللهجات العربية الطبية

**النسخة النهائية المرتبة: البيانات → القياس قبل التدريب → LoRA V1 → تحسين البيانات → RafiQ V2 → الديمو الصوتي**

> ⭐ **للحكّام:** الخلايا التي تحمل علامة **⭐ للحكّام** هي أسرع مسار لمراجعة جودة النموذج ونتائج المشروع.
> تم الإبقاء على أهم مخرجات التشغيل الناجحة حتى تظهر النتائج مباشرة دون الحاجة إلى إعادة تدريب النموذج ذي 4 مليارات باراميتر.

### ⭐ المسار السريع للحكّام
1. ⭐ نتائج النموذج قبل التدريب
2. ⭐ مقارنة الأداء قبل التدريب وبعد V1
3. ⭐ المقارنة النهائية بين النموذج الأساسي وV1 وV2
4. ⭐ نتيجة تجربة المسار الصوتي

**المهمة الأساسية:** تحويل التعبير الصحي باللهجة العامية إلى عربية أبسط وواضحة مع الحفاظ على المعنى، ومن دون إضافة تشخيص أو أعراض أو مدة أو شدة أو عضو غير مذكور في النص الأصلي.


## 1. تجهيز بيئة العمل
تثبيت المكتبات المطلوبة، ربط Google Drive، والتأكد من توفر الـGPU. تدريب وتشغيل ASL-4B مخصصان لبيئة تحتوي على GPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# المكتبات الأساسية المستخدمة لتحميل النموذج، وتدريب LoRA، والتقييم، ومعالجة البيانات
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets openpyxl evaluate rouge_score sacrebleu
print("تم تجهيز المكتبات ✅")


In [ ]:
import torch
print("GPU متاح؟", torch.cuda.is_available())
if torch.cuda.is_available():
    print("اسم الـ GPU:", torch.cuda.get_device_name(0))


## 2. تحميل الداتا سيت
رفع ملف Excel الخاص بداتا سيت RafiQ. يقرأ النوتبوك ورقة `publish_ready` ويستخدم الحقول المنشورة التالية: `split` و`group_id` و`dialect_text` و`simple_arabic` و`medical_department`.


In [ ]:
from google.colab import files
uploaded = files.upload()
DATA_FILENAME = list(uploaded.keys())[0]
print("تم رفع الملف:", DATA_FILENAME)


In [ ]:
import pandas as pd

df = pd.read_excel(
    DATA_FILENAME,
    sheet_name="publish_ready"
)

COL_DIALECT = "dialect_text"
COL_SIMPLE = "simple_arabic"
COL_GROUP = "group_id"
COL_DEPT = "medical_department"

print("عدد الصفوف الكلي:", len(df))

print("\nتوزيع الـ Split:")
print(df["split"].value_counts())

print("\nتوزيع الأقسام الطبية:")
print(df[COL_DEPT].value_counts())

print("\nعدد الـ Groups:")
print(df[COL_GROUP].nunique())

df.head()

## 3. تنظيف وتجهيز النص العربي
توحيد اختلافات الكتابة البسيطة في النص العربي مع الحفاظ على المعنى الكامل لكل شكوى.


In [ ]:
# دالة تنظيف موحدة تُستخدم في التدريب والتقييم
import re

def clean_arabic_text(text):
    text = str(text)
    # 1) شيل التشكيل (فتحة/ضمة/كسرة/شدة/سكون/تنوين) + التطويل (ـ)
    text = re.sub(r'[\u064B-\u0652\u0670\u0640]', '', text)
    # 2) توحيد أشكال الألف والياء
    text = re.sub(r'[إأآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    # 3) تنظيف المسافات الزايدة
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df[COL_DIALECT + "_clean"] = df[COL_DIALECT].apply(clean_arabic_text)
df[COL_SIMPLE + "_clean"]  = df[COL_SIMPLE].apply(clean_arabic_text)

# نستخدم الأعمدة النظيفة من الحين فصاعدًا
COL_DIALECT_CLEAN = COL_DIALECT + "_clean"
COL_SIMPLE_CLEAN  = COL_SIMPLE + "_clean"

print("مقارنة قبل/بعد التنظيف على أول 5 أمثلة:\n")
for i in range(min(5, len(df))):
    print("قبل :", df[COL_DIALECT].iloc[i])
    print("بعد :", df[COL_DIALECT_CLEAN].iloc[i])
    print("-" * 40)



## 4. فحص جودة البيانات
التأكد من حجم الداتا سيت، والقيم الناقصة، والتكرارات المتطابقة، والأهم: التأكد من أن `group_id` الواحد لا ينتقل بين التدريب والتحقق والاختبار.


In [ ]:
# Quality gate before training
print("===== فحص جودة البيانات =====")

print("عدد الصفوف:", len(df))

print(
    "القيم الناقصة:",
    df[
        ["split", COL_GROUP, COL_DIALECT, COL_SIMPLE, COL_DEPT]
    ].isna().sum().sum()
)

print(
    "التكرارات المتطابقة:",
    df.duplicated(
        subset=[COL_DIALECT, COL_SIMPLE, COL_DEPT]
    ).sum()
)

group_split_counts = df.groupby(COL_GROUP)["split"].nunique()

print(
    "Groups الموجودة في أكثر من Split:",
    (group_split_counts > 1).sum()
)

print("\n✅ انتهى فحص جودة الداتا")


## 5. تجهيز الـTokenizer وصيغة التعليمات
تحميل Tokenizer الخاص بـASL-4B وتجهيز نفس صيغة System / User / Assistant المستخدمة أثناء التدريب الموجّه.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "saai-sa/ASL-4B-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# مثال: شوف شكل التوكنز لجملة وحدة
sample_text = df[COL_DIALECT_CLEAN].iloc[0]
tokens = tokenizer.tokenize(sample_text)
print("الجملة:", sample_text)
print("عدد التوكنز:", len(tokens))
print("التوكنز:", tokens)
print()

# توزيع أطوال كل الجمل (عامية + مبسطة) بالتوكنز — يساعدنا نحدد max_seq_length
lengths = df[COL_DIALECT_CLEAN].apply(lambda t: len(tokenizer.tokenize(t)))
lengths_simple = df[COL_SIMPLE_CLEAN].apply(lambda t: len(tokenizer.tokenize(t)))
print("أطول جملة عامية (توكنز):", lengths.max())
print("أطول جملة مبسطة (توكنز):", lengths_simple.max())
print("متوسط طول الجملة العامية:", round(lengths.mean(), 1))


In [ ]:
SYSTEM_PROMPT = (
    "أنت مساعد متخصص في تطبيع اللهجات العربية الطبية إلى عربية بسيطة وواضحة. "
    "مهمتك إعادة صياغة كلام المستخدم فقط، وليس تشخيص حالته. "
    "حافظ على جميع المعلومات الموجودة في النص الأصلي مثل العرض، المكان، الزمن، الشدة والسياق. "
    "لا تضف أي عرض أو تشخيص أو مدة أو شدة أو عضو لم يذكره المستخدم. "
    "لا تحذف أي معلومة موجودة في النص الأصلي. "
    "حوّل الكلمات والتعابير اللهجية الصعبة إلى عربية بسيطة مع الحفاظ على المعنى نفسه."
)

def build_chat_text(dialect_text, simple_text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": str(dialect_text)
        },
        {
            "role": "assistant",
            "content": str(simple_text)
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

df["text"] = df.apply(
    lambda r: build_chat_text(
        r[COL_DIALECT_CLEAN],
        r[COL_SIMPLE_CLEAN]
    ),
    axis=1
)

print(df["text"].iloc[0])

## 6. تقسيم البيانات إلى تدريب وتحقق واختبار
استخدام التقسيمات الموجودة مسبقًا داخل الداتا سيت. تظل مجموعة الاختبار معزولة ولا تُستخدم في التدريب، حتى تكون جميع المقارنات عادلة.


In [ ]:
from datasets import Dataset

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "validation"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df[["text"]])
val_ds   = Dataset.from_pandas(val_df[["text"]])

print("عدد أمثلة التدريب:", len(train_ds))
print("عدد أمثلة التحقق:", len(val_ds))
print("عدد أمثلة الاختبار:", len(test_df))


## 7. ⭐ للحكّام — تقييم النموذج قبل التدريب
يتم تقييم نموذج ASL-4B الأصلي على نفس مجموعة الاختبار المعزولة التي ستُستخدم لاحقًا مع V1 وV2. هذه النتائج تمثل **خط الأساس قبل التدريب**.


In [ ]:
def clean_model_output(text):
    """الاحتفاظ بأول إجابة مولدة فقط وإزالة أي رموز أو بقايا خاصة بالمحادثة أو النموذج."""
    text = str(text).strip()
    stop_markers = [
        "\nuser", "\nassistant", "<think>",
        "<|user|>", "<|assistant|>", "<|end|>"
    ]
    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()
    return text


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
# دالة توليد موحدة تُستخدم لتقييم النموذج الأساسي وV1 وV2
def generate_with_model(model, dialect_text, max_new_tokens=100):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": dialect_text},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return clean_model_output(result)


In [ ]:
sample_input = test_df[COL_DIALECT_CLEAN].iloc[0]

sample_reference = test_df[COL_SIMPLE_CLEAN].iloc[0]

sample_prediction = generate_with_model(
    baseline_model,
    sample_input
)

print("الجملة العامية:")
print(sample_input)

print("\nالإجابة الصحيحة في الداتا:")
print(sample_reference)

print("\nجواب المودل قبل التدريب:")
print(sample_prediction)

In [ ]:
baseline_predictions = []

for i in range(len(test_df)):

    dialect_text = test_df[COL_DIALECT_CLEAN].iloc[i]

    pred = generate_with_model(
        baseline_model,
        dialect_text
    )

    baseline_predictions.append(pred)

    print(
        f"تم {i + 1}/{len(test_df)}"
    )


test_df["baseline_prediction"] = baseline_predictions

print("\nانتهى تقييم النموذج قبل التدريب بالكامل ✅")

In [ ]:
# ⭐ للحكّام: BASELINE METRICS (before fine-tuning)
import evaluate
from sacrebleu.metrics import CHRF

bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")
chrf_metric = CHRF()

test_df["baseline_prediction_clean"] = (
    test_df["baseline_prediction"].apply(clean_model_output)
)

predictions_before = test_df["baseline_prediction_clean"].tolist()
references_text = test_df[COL_SIMPLE_CLEAN].tolist()
references_bleu = [[x] for x in references_text]

bleu_before = bleu.compute(predictions=predictions_before, references=references_bleu)
rouge_before = rouge.compute(
    predictions=predictions_before,
    references=references_text,
    tokenizer=lambda x: x.split()
)
chrf_before = chrf_metric.corpus_score(predictions_before, [references_text]).score

print("===== CLEAN BASELINE =====")
print("BLEU:", round(bleu_before["score"], 2))
print("ROUGE-L:", round(rouge_before["rougeL"], 4))
print("chrF:", round(chrf_before, 2))


In [ ]:
# Save baseline predictions for reproducibility
baseline_results = test_df[[
    "group_id", COL_DIALECT, COL_SIMPLE, "baseline_prediction_clean"
]].copy()
baseline_results.to_csv("baseline_test_predictions.csv", index=False, encoding="utf-8-sig")
print("Baseline predictions saved ✅")


In [ ]:
del baseline_model

import gc
gc.collect()

torch.cuda.empty_cache()

print("تم حذف Baseline من الـGPU ✅")

## 8. التدريب الأول باستخدام LoRA — النسخة V1
تدريب LoRA لأول مرة على بيانات التدريب الأصلية، ثم تقييم النسخة الناتجة على نفس مجموعة الاختبار المستخدمة في خط الأساس.


In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit config جاهز بـ float16 ✅")

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

print("تم تحميل المودل بـ float16 ✅")

In [ ]:
# LoRA setup; trainable adapter parameters are cast to float32 to avoid the earlier BF16 AMP error
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
)

model = get_peft_model(model, lora_config)

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()


In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./rafeeq_asl_lora_final"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=42,
    data_seed=42,

    fp16=False,
    bf16=False,

    report_to="none",

    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print("Trainer أعيد إنشاؤه بعد تصحيح dtype ✅")

In [ ]:
# Train V1
train_result = trainer.train()


In [ ]:
import pandas as pd

history_df = pd.DataFrame(
    trainer.state.log_history
)

history_df.to_csv(
    "training_history.csv",
    index=False,
    encoding="utf-8-sig"
)

display(history_df)

In [ ]:
eval_history = history_df[
    history_df["eval_loss"].notna()
].copy()

columns_to_show = [
    col for col in [
        "epoch",
        "eval_loss",
        "eval_entropy",
        "eval_mean_token_accuracy"
    ]
    if col in eval_history.columns
]

display(
    eval_history[columns_to_show]
)

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("تم حفظ أفضل LoRA في:", OUTPUT_DIR)

In [ ]:
import shutil
import os

drive_path = "/content/drive/MyDrive/rafeeq_asl_lora_final"

if os.path.exists(drive_path):
    shutil.rmtree(drive_path)

shutil.copytree(
    OUTPUT_DIR,
    drive_path
)

print("تم حفظ المودل النهائي في Google Drive ✅")

### V1 Evaluation
Evaluate the trained V1 adapter on the exact same held-out test examples used by the baseline.


In [ ]:
after_predictions = []

for i in range(len(test_df)):

    dialect_text = test_df[COL_DIALECT_CLEAN].iloc[i]

    pred = generate_with_model(
        model,
        dialect_text
    )

    after_predictions.append(pred)

    print(f"تم {i + 1}/{len(test_df)}")

test_df["after_finetune_prediction"] = after_predictions

print("\nانتهى اختبار المودل النهائي ✅")

In [ ]:
# V1 corpus-level metrics
predictions_after = (
    test_df["after_finetune_prediction"]
    .tolist()
)

references_text = (
    test_df[COL_SIMPLE_CLEAN]
    .tolist()
)

references_bleu = [
    [x]
    for x in references_text
]

bleu_after = bleu.compute(
    predictions=predictions_after,
    references=references_bleu
)

rouge_after = rouge.compute(
    predictions=predictions_after,
    references=references_text,
    tokenizer=lambda x: x.split()
)

chrf_after = chrf_metric.corpus_score(
    predictions_after,
    [references_text]
).score

print("===== AFTER FINE-TUNING =====")
print("BLEU:", round(bleu_after["score"], 2))
print("ROUGE-L:", round(rouge_after["rougeL"], 4))
print("chrF:", round(chrf_after, 2))


In [ ]:
# V1 token overlap and exact-match metrics
from collections import Counter

def token_f1(reference, prediction):

    ref_tokens = reference.split()
    pred_tokens = prediction.split()

    ref_counts = Counter(ref_tokens)
    pred_counts = Counter(pred_tokens)

    common = sum(
        (ref_counts & pred_counts).values()
    )

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


token_f1_scores = [
    token_f1(ref, pred)
    for ref, pred in zip(
        references_text,
        predictions_after
    )
]

token_f1_after = (
    sum(token_f1_scores)
    / len(token_f1_scores)
)

exact_matches = [
    int(
        clean_arabic_text(pred)
        == clean_arabic_text(ref)
    )
    for pred, ref in zip(
        predictions_after,
        references_text
    )
]

exact_match_accuracy = (
    sum(exact_matches)
    / len(exact_matches)
)

print("Token F1:", round(token_f1_after, 4))
print("Exact Match Accuracy:", round(exact_match_accuracy, 4))


### ⭐ للحكّام — مقارنة الأداء قبل التدريب وبعد النسخة الأولى V1
This is the first clear before/after comparison on the same test set.


In [ ]:
# ⭐ للحكّام: BASELINE vs V1 COMPARISON
results = pd.DataFrame({
    "Metric": [
        "BLEU",
        "ROUGE-L",
        "chrF",
        "Token F1",
        "Exact Match Accuracy"
    ],

    "Baseline": [
        round(bleu_before["score"], 2),
        round(rouge_before["rougeL"], 4),
        round(chrf_before, 2),
        None,
        None
    ],

    "Fine-tuned": [
        round(bleu_after["score"], 2),
        round(rouge_after["rougeL"], 4),
        round(chrf_after, 2),
        round(token_f1_after, 4),
        round(exact_match_accuracy, 4)
    ]
})

display(results)


In [ ]:
test_results = test_df[
    [
        "group_id",
        COL_DIALECT,
        COL_SIMPLE,
        "baseline_prediction_clean",
        "after_finetune_prediction"
    ]
].copy()

test_results["token_f1"] = token_f1_scores
test_results["exact_match"] = exact_matches

test_results.to_csv(
    "final_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("تم حفظ نتائج الـTest ✅")

In [ ]:
# Free V1 GPU memory before loading a fresh 4-bit base model for V2
import gc
try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("V1 GPU memory released ✅")


## 9. Targeted Data Augmentation for V2
Add the curated improvement examples to the **training split only**. Validation and test data are not augmented.


In [ ]:
from google.colab import files

uploaded_v2 = files.upload()
AUG_FILENAME = next(iter(uploaded_v2.keys()))
print("Improvement file uploaded:", AUG_FILENAME)


In [ ]:
import pandas as pd

augmentation_df = pd.read_excel(
    AUG_FILENAME,
    sheet_name="Augmentation_Data"
)

print("عدد أمثلة التحسين:", len(augmentation_df))
print("الأعمدة:", augmentation_df.columns.tolist())
display(augmentation_df.head())


In [ ]:
augmentation_train = augmentation_df[
    ["group_id", "dialect_text", "simple_arabic", "medical_department"]
].copy()

augmentation_train["split"] = "train"

# نخلي ترتيب الأعمدة مطابق للداتا الأصلية
augmentation_train = augmentation_train[
    ["split", "group_id", "dialect_text", "simple_arabic", "medical_department"]
]

print("عدد الأمثلة:", len(augmentation_train))
print("القيم الناقصة:", augmentation_train.isna().sum().sum())
print("التقسيم:", augmentation_train["split"].value_counts().to_dict())

display(augmentation_train.head())

In [ ]:
train_original = df[df["split"] == "train"][
    ["split", "group_id", "dialect_text", "simple_arabic", "medical_department"]
].copy()

train_v2 = pd.concat(
    [train_original, augmentation_train],
    ignore_index=True
)

print("Train الأصلي:", len(train_original))
print("أمثلة التحسين:", len(augmentation_train))
print("Train V2 الجديد:", len(train_v2))
print("القيم الناقصة:", train_v2.isna().sum().sum())

In [ ]:
from datasets import Dataset

train_v2_ds = Dataset.from_pandas(
    train_v2[
        ["dialect_text", "simple_arabic"]
    ].copy(),
    preserve_index=False
)

print(train_v2_ds)
print("عدد أمثلة Train V2:", len(train_v2_ds))

In [ ]:
def format_training_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["dialect_text"]},
        {"role": "assistant", "content": example["simple_arabic"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


train_v2_ds = train_v2_ds.map(format_training_example)

print(train_v2_ds)
print("\nأول مثال جاهز للتدريب:\n")
print(train_v2_ds[0]["text"])

## 10. تدريب RafiQ V2
Train a fresh LoRA adapter on the original training data plus the targeted augmentation examples.


In [ ]:
# Fresh ASL-4B base + LoRA for V2
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

MODEL_NAME = "saai-sa/ASL-4B-v1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_v2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model_v2 = prepare_model_for_kbit_training(model_v2)

lora_config_v2 = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model_v2 = get_peft_model(model_v2, lora_config_v2)

# نفس حل مشكلة BF16 اللي واجهتنا قبل
for name, param in model_v2.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

model_v2.print_trainable_parameters()


In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR_V2 = "./rafeeq_asl_lora_v2"

training_args_v2 = SFTConfig(
    output_dir=OUTPUT_DIR_V2,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=42,
    data_seed=42,

    fp16=False,
    bf16=False,

    report_to="none",

    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer_v2 = SFTTrainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_v2_ds,
    eval_dataset=val_ds,
)

print("Trainer V2 جاهز ✅")
print("Train:", len(train_v2_ds))
print("Validation:", len(val_ds))

In [ ]:
# تدريب النسخة V2
train_result_v2 = trainer_v2.train()


## 11. ⭐ للحكّام — التقييم النهائي للنسخة V2
يتم تقييم V2 على **نفس مجموعة الاختبار المعزولة المكوّنة من 49 مثالًا** التي استُخدمت مع النموذج الأساسي وV1.


In [ ]:
v2_predictions = []

for i in range(len(test_df)):
    dialect_text = test_df[COL_DIALECT_CLEAN].iloc[i]

    pred = generate_with_model(
        model_v2,
        dialect_text
    )

    v2_predictions.append(pred)

    print(f"تم {i + 1}/{len(test_df)}")

test_df["v2_prediction"] = v2_predictions

print("تم اختبار V2 على جميع أمثلة الـTest ✅")

In [ ]:
# ⭐ للحكّام: FINAL V2 METRICS
predictions_v2 = test_df["v2_prediction"].tolist()
references_text = test_df[COL_SIMPLE_CLEAN].tolist()
references_bleu = [[x] for x in references_text]

bleu_v2 = bleu.compute(
    predictions=predictions_v2,
    references=references_bleu
)

rouge_v2 = rouge.compute(
    predictions=predictions_v2,
    references=references_text,
    tokenizer=lambda x: x.split()
)

chrf_v2 = chrf_metric.corpus_score(
    predictions_v2,
    [references_text]
).score

token_f1_scores_v2 = [
    token_f1(ref, pred)
    for ref, pred in zip(references_text, predictions_v2)
]

token_f1_v2 = sum(token_f1_scores_v2) / len(token_f1_scores_v2)

exact_matches_v2 = [
    int(clean_arabic_text(pred) == clean_arabic_text(ref))
    for pred, ref in zip(predictions_v2, references_text)
]

exact_match_v2 = sum(exact_matches_v2) / len(exact_matches_v2)

print("V2 BLEU:", round(bleu_v2["score"], 2))
print("V2 ROUGE-L:", round(rouge_v2["rougeL"], 4))
print("V2 chrF:", round(chrf_v2, 2))
print("V2 Token F1:", round(token_f1_v2, 4))
print("V2 Exact Match:", round(exact_match_v2, 4))


In [ ]:
v2_results = test_df[
    [
        "group_id",
        COL_DIALECT,
        COL_SIMPLE,
        "baseline_prediction_clean",
        "after_finetune_prediction",
        "v2_prediction"
    ]
].copy()

v2_results["token_f1_v2"] = token_f1_scores_v2
v2_results["exact_match_v2"] = exact_matches_v2

v2_results.to_csv(
    "v2_final_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("تم إنشاء الملف ✅")
print("عدد الأمثلة:", len(v2_results))

## 12. حفظ نموذج RafiQ V2 النهائي والنتائج
حفظ LoRA Adapter النهائي وملفات نتائج التقييم في Google Drive حتى يمكن استخدام النموذج لاحقًا دون إعادة التدريب.


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

V2_DRIVE_DIR = "/content/drive/MyDrive/rafeeq_asl_lora_v2"
os.makedirs(V2_DRIVE_DIR, exist_ok=True)

trainer_v2.save_model(V2_DRIVE_DIR)
tokenizer.save_pretrained(V2_DRIVE_DIR)

print("تم حفظ مودل V2 في Google Drive ✅")
print(V2_DRIVE_DIR)

In [ ]:
import os
import pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/rafeeq_v2_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# حفظ التوقعات لكل مثال على حدة
v2_results.to_csv(
    f"{RESULTS_DIR}/v2_final_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# حفظ المقاييس من القيم المحسوبة فعليًا أعلاه دون كتابة أرقام ثابتة يدويًا
v2_metrics = pd.DataFrame({
    "Metric": ["BLEU", "ROUGE-L", "chrF", "Token F1", "Exact Match"],
    "V2": [
        round(bleu_v2["score"], 2),
        round(rouge_v2["rougeL"], 4),
        round(chrf_v2, 2),
        round(token_f1_v2, 4),
        round(exact_match_v2, 4),
    ]
})
v2_metrics.to_csv(f"{RESULTS_DIR}/v2_metrics.csv", index=False, encoding="utf-8-sig")

# حفظ سجل التدريب
history_v2 = pd.DataFrame(trainer_v2.state.log_history)
history_v2.to_csv(f"{RESULTS_DIR}/v2_training_history.csv", index=False, encoding="utf-8-sig")

print("تم حفظ كل نتائج V2 في Google Drive ✅")
print(RESULTS_DIR)


## ⭐ للحكّام — المقارنة النهائية قبل وبعد التدريب

جميع الصفوف التالية تم حسابها باستخدام **نفس مجموعة الاختبار المعزولة** وبنفس طريقة التقييم، لذلك المقارنة مباشرة وعادلة.

| مرحلة النموذج | BLEU | ROUGE-L | chrF |
|---|---:|---:|---:|
| **النموذج الأساسي قبل التدريب** | **0.70** | **0.1189** | **23.82** |
| **LoRA V1** | **40.83** | **0.6368** | **61.55** |
| **RafiQ V2 النهائي** | **68.44** | **0.7784** | **79.41** |

مقاييس إضافية للنسخة النهائية V2: **Token F1 = 0.7768** و **Exact Match = 0.4898**.

> ⭐ هذه هي أهم خلية لعرض تحسن أداء النموذج أمام الحكّام.


## 13. تجربة النموذج على نص جديد
لاستخدام النموذج في الديمو أو عند التشغيل لاحقًا، يتم تحميل V2 المحفوظ فقط إذا لم يكن موجودًا بالفعل في الذاكرة، ثم يمكن تمرير شكاوى جديدة إليه.


In [ ]:
# تحميل النموذج المحفوظ عند بدء جلسة جديدة، عند الحاجة فقط
# إذا كان V2 موجودًا بالفعل في الذاكرة بعد التدريب فلن تتم إعادة تحميله
if "model_v2" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    MODEL_NAME = "saai-sa/ASL-4B-v1"
    V2_DRIVE_DIR = "/content/drive/MyDrive/rafeeq_asl_lora_v2"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(V2_DRIVE_DIR)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model_v2 = PeftModel.from_pretrained(base_model, V2_DRIVE_DIR)

model_v2.eval()
print("RafiQ V2 جاهز للتجربة ✅")


In [ ]:
def generate_for_deployment(model, text, max_new_tokens=64):
    """تعليمات مختصرة تُستخدم في النسخة النهائية لتجربة النص والصوت."""
    prompt = f"""حوّل الشكوى الصحية التالية من اللهجة العامية إلى عربية مبسطة وواضحة مع الحفاظ على المعنى وعدم إضافة تشخيص أو أعراض جديدة.

الشكوى:
{text}

العربية المبسطة:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return generated.strip()

def normalize_complaint(dialect_text: str) -> str:
    # تنظيف نص المستخدم ثم تمريره إلى النسخة النهائية V2
    cleaned_input = clean_arabic_text(dialect_text)
    return generate_for_deployment(model_v2, cleaned_input)

print("دوال تجربة النص جاهزة ✅")


In [ ]:
# تجربة نصية نهائية بسيطة
test_text = "بطني يلويني"
text_result = normalize_complaint(test_text)
print("النص الأصلي:")
print(test_text)
print("\nRafiQ V2:")
print(text_result)


## 14. تجربة المسار الصوتي — Whisper + RafiQ V2

**الصوت → Whisper لتحويل الكلام إلى نص → النص اللهجي → RafiQ V2 → العربية المبسطة**

يُستخدم Whisper كنموذج جاهز لتحويل الصوت إلى نص، ولا يتم عمل Fine-tuning له داخل هذا النوتبوك.


In [ ]:
# تثبيت مكتبات الصوت المطلوبة للنموذج الأولي
!pip install -q -U librosa soundfile
!apt-get -qq install -y ffmpeg
print("تم تجهيز مكتبات الصوت ✅")


In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    dtype=dtype,
    device=device
)

print("Whisper جاهز ✅")
print("الجهاز المستخدم:", "GPU" if torch.cuda.is_available() else "CPU")


In [ ]:
def voice_to_simple_arabic(audio_path):
    # المرحلة 1: تحويل الصوت إلى نص عربي باستخدام Whisper
    stt_result = whisper_pipe(
        audio_path,
        generate_kwargs={
            "language": "arabic",
            "task": "transcribe"
        }
    )
    dialect_text = stt_result["text"].strip()

    # المرحلة 2: تحويل النص اللهجي إلى عربية مبسطة باستخدام RafiQ V2
    simple_text = normalize_complaint(dialect_text)

    return {
        "whisper_text": dialect_text,
        "simple_text": simple_text
    }

print("المسار الصوتي لـRafiQ جاهز ✅")


In [ ]:
from google.colab import files

uploaded_audio = files.upload()
audio_filename = next(iter(uploaded_audio.keys()))
print("تم رفع الملف الصوتي ✅")
print("اسم الملف:", audio_filename)


In [ ]:
# ⭐ للحكّام: RUN THE COMPLETE VOICE PIPELINE
result = voice_to_simple_arabic(audio_filename)

print("🎙️ Whisper:")
print(result["whisper_text"])
print("\n🩺 RafiQ V2:")
print(result["simple_text"])


### ⭐ للحكّام — نتيجة الديمو الصوتي
أنتج التشغيل الناجح للنموذج الأولي النتيجة التالية:

**الكلام المنطوق → Whisper:** `بطني يلويني`  
**RafiQ V2 → العربية المبسطة:** `بطني يؤلمني`


In [ ]:
# ⭐ للحكّام: clean presentation cell
print("===== تجربة المسار الصوتي لـRafiQ =====")
print("🎙️ النص المستخرج من الصوت:")
print(result["whisper_text"])
print("\n🩺 النص بعد التبسيط:")
print(result["simple_text"])
print("\n✅ تم تشغيل المسار الصوتي بنجاح")


## ✅ ملخص النوتبوك النهائي

يوثّق هذا النوتبوك مراحل تطوير RafiQ كاملة وبترتيب واضح:

1. تحميل الداتا سيت وتنظيف النص العربي
2. فحص جودة البيانات والتأكد من عدم وجود تسرب بين التقسيمات
3. تقييم ASL-4B قبل التدريب
4. تدريب وتقييم LoRA V1
5. زيادة بيانات التدريب بشكل مستهدف
6. تدريب النسخة المحسنة RafiQ V2
7. التقييم النهائي على نفس مجموعة الاختبار المعزولة
8. حفظ LoRA Adapter النهائي وملفات النتائج
9. تجربة النموذج على نص جديد
10. بناء نموذج أولي صوتي باستخدام Whisper + RafiQ V2

### أهم النتائج النهائية
- قبل التدريب: BLEU **0.70**، ROUGE-L **0.1189**، chrF **23.82**
- V1: BLEU **40.83**، ROUGE-L **0.6368**، chrF **61.55**
- V2: BLEU **68.44**، ROUGE-L **0.7784**، chrF **79.41**، Token F1 **0.7768**، Exact Match **0.4898**
- الديمو الصوتي: `بطني يلويني` → `بطني يؤلمني`

> جميع النتائج الرئيسية أعلاه من مخرجات تشغيل ناجحة موجودة في النوتبوك الأصلي، ولم يتم تضمين مخرجات المحاولات الفاشلة أو خلايا التصحيح.
